# Sarcasm Detection Capstone — Exploratory Data Analysis

This notebook explores the raw dataset before any modeling: class balance, headline
length distributions, vocabulary differences between sarcastic and non-sarcastic
headlines, and duplicate/leakage checks. Run this **before** `src/train_method1_scratch.py`
etc. — it explains *why* certain preprocessing decisions were made in `src/data_utils.py`.

Run this notebook from the project root (open the whole `sarcasm-capstone` folder in
VS Code, not just `notebooks/`) so the relative paths below resolve correctly.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from data_utils import load_combined_dataset, clean_text

pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (8, 4)

## 1. Load the raw data

In [ ]:
df = load_combined_dataset()
print(df.shape)
df.head()

## 2. Class balance

The dataset needs to be reasonably balanced for accuracy to be a fair headline metric
(as opposed to needing precision/recall/F1 to tell the real story on an imbalanced set).

In [ ]:
counts = df["is_sarcastic"].value_counts().sort_index()
print(counts)
print(f"\nSarcastic %: {df['is_sarcastic'].mean():.1%}")

counts.plot(kind="bar", color=["#2E7D7B", "#1F3864"])
plt.xticks([0, 1], ["Not sarcastic", "Sarcastic"], rotation=0)
plt.title("Class balance")
plt.ylabel("Count")
plt.show()

## 3. Headline length distribution

This informs the `MAX_LEN` (padding length) choice in `train_method1_scratch.py` —
we want a length that covers the vast majority of headlines without excessive padding.

In [ ]:
df["word_count"] = df["headline"].str.split().apply(len)

print(df["word_count"].describe())
print(f"\n95th percentile: {df['word_count'].quantile(0.95):.0f} words")
print(f"99th percentile: {df['word_count'].quantile(0.99):.0f} words")

df["word_count"].hist(bins=30)
plt.title("Headline length distribution (words)")
plt.xlabel("Word count")
plt.ylabel("Frequency")
plt.show()

In [ ]:
fig, ax = plt.subplots()
df[df["is_sarcastic"] == 0]["word_count"].hist(bins=30, alpha=0.6, label="Not sarcastic", ax=ax)
df[df["is_sarcastic"] == 1]["word_count"].hist(bins=30, alpha=0.6, label="Sarcastic", ax=ax)
ax.set_title("Headline length by class")
ax.set_xlabel("Word count")
ax.legend()
plt.show()

## 4. Most common words per class

A quick lexical sanity check: sarcastic (Onion-style) headlines often lean on absurdist,
hyperbolic phrasing ("area man", "nation's X"), while non-sarcastic headlines lean on
concrete nouns/entities. This is exactly the kind of surface pattern a from-scratch
Word2Vec + BiLSTM model can partially pick up on — and exactly what a transformer or
LLM is expected to generalize beyond.

In [ ]:
from nltk.corpus import stopwords
import nltk
try:
    stopwords.words("english")
except LookupError:
    nltk.download("stopwords", quiet=True)

STOP = set(stopwords.words("english"))

def top_words(subset_df, n=20):
    tokens = []
    for h in subset_df["headline"]:
        tokens.extend([w for w in clean_text(h).split() if w not in STOP and len(w) > 2])
    return Counter(tokens).most_common(n)

print("Top words — SARCASTIC:")
for word, cnt in top_words(df[df["is_sarcastic"] == 1]):
    print(f"  {word:20s} {cnt}")

print("\nTop words — NOT SARCASTIC:")
for word, cnt in top_words(df[df["is_sarcastic"] == 0]):
    print(f"  {word:20s} {cnt}")

## 5. Duplicate check (leakage risk)

Confirms why `data_utils.py` deduplicates on exact headline text before splitting —
identical headlines appearing in both train and test would let a model "memorize"
rather than generalize, inflating test accuracy artificially.

In [ ]:
raw_v2_len = len(df)  # already deduplicated by load_combined_dataset()
dupes = df["headline"].duplicated().sum()
print(f"Duplicate headlines remaining after dedup: {dupes}")
print(f"Total unique headlines: {raw_v2_len}")

## 6. Sample headlines from each class

A manual read of a few examples — useful for sanity-checking that the labels make
intuitive sense before trusting any model's output.

In [ ]:
print("SARCASTIC examples:")
for h in df[df["is_sarcastic"] == 1]["headline"].sample(5, random_state=1):
    print(f"  - {h}")

print("\nNOT SARCASTIC examples:")
for h in df[df["is_sarcastic"] == 0]["headline"].sample(5, random_state=1):
    print(f"  - {h}")

## 7. Takeaways feeding into modeling decisions

- Classes are well balanced (~47-53%), so **accuracy is a valid primary metric**
  (confirmed quantitatively above, not just assumed).
- Headline length is short and consistent enough that a **short padding length
  (~30 tokens)** loses very little information — confirmed by the percentile check.
- Sarcastic headlines show a distinct absurdist/hyperbolic lexical style, which
  supports trying a **from-scratch embedding approach** (Method 1) alongside
  transfer learning and LLM approaches, since even shallow word-level signal
  carries real information here.
- Deduplication before splitting is **necessary**, not optional, to get a trustworthy
  test accuracy number.

Proceed to `src/data_utils.py` to generate the official train/test split, then run
each method's training script.